In [4]:
# Library for Data Manipulation
import numpy as np
import pandas as pd

#Library for Data Visualization.
import seaborn as sns
import matplotlib.pyplot as plt


%matplotlib inline
sns.set_style("whitegrid")
plt.style.use("fivethirtyeight")

# Library for Statistical Modelling
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


In [5]:
employee_data = pd.read_csv(r'IBM-HR-Analytics-Employee-Attrition-and-Performance-Revised.csv')

In [6]:
# Print top 5 rows in the dataframe.
employee_data.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,College,Life Sciences,Medium,Female,...,Excellent,Low,0,8,0,Bad,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,Below College,Life Sciences,High,Male,...,Outstanding,Very High,1,10,3,Better,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,College,Other,Very High,Male,...,Excellent,Medium,0,7,3,Better,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,Master,Life Sciences,Very High,Female,...,Excellent,High,0,8,3,Better,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,Below College,Medical,Low,Male,...,Excellent,Very High,1,6,3,Better,2,2,2,2


In [7]:
# Print bottom 5 rows in the dataframe.
employee_data.tail()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
1465,36,No,Travel_Frequently,884,Research & Development,23,College,Medical,High,Male,...,Excellent,High,1,17,3,Better,5,2,0,3
1466,39,No,Travel_Rarely,613,Research & Development,6,Below College,Medical,Very High,Male,...,Excellent,Low,1,9,5,Better,7,7,1,7
1467,27,No,Travel_Rarely,155,Research & Development,4,Bachelor,Life Sciences,Medium,Male,...,Outstanding,Medium,1,6,0,Better,6,2,0,3
1468,49,No,Travel_Frequently,1023,Sales,2,Bachelor,Medical,Very High,Male,...,Excellent,Very High,0,17,3,Good,9,6,0,8
1469,34,No,Travel_Rarely,628,Research & Development,8,Bachelor,Medical,Medium,Male,...,Excellent,Low,0,6,3,Best,4,3,1,2


In [8]:
# Print the shape of the DataFrame
print("The shape of data frame:", employee_data.shape)
# Print the length (number of rows) of the DataFrame
print("Number of Rows in the dataframe:", len(employee_data))
# Print the number of columns in the DataFrame
print("Number of Columns in the dataframe:", len(employee_data.columns))

The shape of data frame: (1470, 31)
Number of Rows in the dataframe: 1470
Number of Columns in the dataframe: 31


In [9]:
# Encode target variable (Attrition)
label_encoder = LabelEncoder()
employee_data["Attrition"] = label_encoder.fit_transform(employee_data["Attrition"])
print(f"Attrition classes: {label_encoder.classes_}")

Attrition classes: ['No' 'Yes']


In [10]:
employee_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   int32 
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   object
 7   EducationField            1470 non-null   object
 8   EnvironmentSatisfaction   1470 non-null   object
 9   Gender                    1470 non-null   object
 10  HourlyRate                1470 non-null   int64 
 11  JobInvolvement            1470 non-null   object
 12  JobLevel                  1470 non-null   object
 13  JobRole                   1470 non-null   object
 14  JobSatisfaction         

In [11]:
# Separate features and target
X = employee_data.drop('Attrition', axis=1)
y = employee_data['Attrition']

# Identify categorical and numerical columns
categorical_cols = [col for col in X.columns if X[col].dtype == 'object']
numerical_cols = [col for col in X.columns if X[col].dtype in ['int64', 'float64']]

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numerical columns: {len(numerical_cols)}")

# One-Hot Encode categorical variables
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dtype='uint8')

# Remove duplicate rows
X_encoded = X_encoded.drop_duplicates().reset_index(drop=True)
y = y[X_encoded.index]

# Remove duplicate columns
X_encoded = X_encoded.loc[:, ~X_encoded.columns.duplicated()]

print(f"\nOriginal shape: {X.shape}")
print(f"After encoding and cleaning: {X_encoded.shape}")

Categorical columns: 15
Numerical columns: 15

Original shape: (1470, 30)
After encoding and cleaning: (1470, 60)


In [12]:
# Display shape info
print(f"Final feature set shape: {X_encoded.shape}")
print(f"Target shape: {y.shape}")
print(f"\nData ready for modeling!")


Final feature set shape: (1470, 60)
Target shape: (1470,)

Data ready for modeling!


In [13]:
# Calculate correlation with target and select features
feature_correlation = X_encoded.corrwith(y).abs().sort_values(ascending=False)
threshold = 0.02
selected_features = feature_correlation[feature_correlation > threshold].index.tolist()

print(f"Total features: {X_encoded.shape[1]}")
print(f"Features selected (correlation > {threshold}): {len(selected_features)}")
print(f"\nTop 10 correlated features:")
print(feature_correlation.head(10))

Total features: 60
Features selected (correlation > 0.02): 46

Top 10 correlated features:
OverTime_Yes                    0.246118
MaritalStatus_Single            0.175419
TotalWorkingYears               0.171063
YearsInCurrentRole              0.160545
MonthlyIncome                   0.159840
Age                             0.159205
JobRole_Sales Representative    0.157234
YearsWithCurrManager            0.156199
StockOptionLevel                0.137145
YearsAtCompany                  0.134392
dtype: float64


In [14]:
# Select features with high correlation
X_final = X_encoded[selected_features].copy()

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, 
    test_size=0.3, 
    random_state=42,
    stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"\nClass distribution in training set:")
print(pd.Series(y_train).value_counts())
print(f"\nClass distribution in testing set:")
print(pd.Series(y_test).value_counts())

Training set shape: (1029, 46)
Testing set shape: (441, 46)

Class distribution in training set:
Attrition
0    863
1    166
Name: count, dtype: int64

Class distribution in testing set:
Attrition
0    370
1     71
Name: count, dtype: int64


In [15]:
# Simple evaluation function - show metrics as numbers only
def simple_evaluate(model, X_train, X_test, y_train, y_test, model_name="Model"):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    print(f"\n{'='*50}")
    print(f"{model_name}")
    print(f"{'='*50}")
    print(f"Training Accuracy: {accuracy_score(y_train, y_train_pred):.4f}")
    print(f"Testing Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Training ROC-AUC: {roc_auc_score(y_train, y_train_pred):.4f}")
    print(f"Testing ROC-AUC: {roc_auc_score(y_test, y_test_pred):.4f}")
    print(f"\nConfusion Matrix (Test):\n{confusion_matrix(y_test, y_test_pred)}")
    print(f"\nClassification Report (Test):\n{classification_report(y_test, y_test_pred)}")
    
    return roc_auc_score(y_test, y_test_pred)

In [16]:
y_test.value_counts()[0] / y_test.shape[0]

0.8390022675736961

In [17]:
stay = (y_train.value_counts()[0] / y_train.shape)[0]
leave = (y_train.value_counts()[1] / y_train.shape)[0]

print("===============TRAIN=================")
print(f"Staying Rate: {stay * 100:.2f}%")
print(f"Leaving Rate: {leave * 100 :.2f}%")

stay = (y_test.value_counts()[0] / y_test.shape)[0]
leave = (y_test.value_counts()[1] / y_test.shape)[0]

print("===============TEST=================")
print(f"Staying Rate: {stay * 100:.2f}%")
print(f"Leaving Rate: {leave * 100 :.2f}%")

===============TRAIN=================
Staying Rate: 83.87%
Leaving Rate: 16.13%
===============TEST=================
Staying Rate: 83.90%
Leaving Rate: 16.10%


In [18]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(solver='liblinear', penalty='l1', random_state=42))
])
lr_pipeline.fit(X_train, y_train)

lr_auc = simple_evaluate(lr_pipeline, X_train, X_test, y_train, y_test, "Logistic Regression")


Logistic Regression
Training Accuracy: 0.9067
Testing Accuracy: 0.8821
Training ROC-AUC: 0.7522
Testing ROC-AUC: 0.7021

Confusion Matrix (Test):
[[358  12]
 [ 40  31]]

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.90      0.97      0.93       370
           1       0.72      0.44      0.54        71

    accuracy                           0.88       441
   macro avg       0.81      0.70      0.74       441
weighted avg       0.87      0.88      0.87       441



In [19]:
rf_pipeline = Pipeline([
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
rf_pipeline.fit(X_train, y_train)

rf_auc = simple_evaluate(rf_pipeline, X_train, X_test, y_train, y_test, "Random Forest")


Random Forest
Training Accuracy: 1.0000
Testing Accuracy: 0.8367
Training ROC-AUC: 1.0000
Testing ROC-AUC: 0.5442

Confusion Matrix (Test):
[[361   9]
 [ 63   8]]

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.85      0.98      0.91       370
           1       0.47      0.11      0.18        71

    accuracy                           0.84       441
   macro avg       0.66      0.54      0.55       441
weighted avg       0.79      0.84      0.79       441



In [21]:
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(kernel='rbf', random_state=42))
])
svm_pipeline.fit(X_train, y_train)

svm_auc = simple_evaluate(svm_pipeline, X_train, X_test, y_train, y_test, "Support Vector Machine")


Support Vector Machine
Training Accuracy: 0.9184
Testing Accuracy: 0.8662
Training ROC-AUC: 0.7494
Testing ROC-AUC: 0.6016

Confusion Matrix (Test):
[[367   3]
 [ 56  15]]

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.87      0.99      0.93       370
           1       0.83      0.21      0.34        71

    accuracy                           0.87       441
   macro avg       0.85      0.60      0.63       441
weighted avg       0.86      0.87      0.83       441

